In [ ]:
### Shared Util functions ###

# Build a tenacity decorator first. 
import tenacity
from urllib.parse import urlparse, urljoin

def give_up_on_failure(retry_state):
    print(f"Giving up after {retry_state.attempt_number} attempts.")
    return "skip"  # Do not retry anymore

retry = tenacity.retry(
    stop=tenacity.stop_after_attempt(10),
    wait=tenacity.wait_exponential(multiplier=1, min=2, max=32),
    retry_error_callback=give_up_on_failure
)


import time
import requests
from urllib.parse import urlencode

@retry
def download_cdx_data(url, sleep=1.5, from_timestamp=None, to_timestamp=None, filters=None, collapse=None):
    time.sleep(sleep)
    params = [('url', url)]
    if from_timestamp:
        params.append(('from', from_timestamp))
    if to_timestamp:
        params.append(('to', to_timestamp))
    if collapse:
        params.append(('collapse', collapse))
    if filters:
        for filter_item in filters:
            params.append(('filter', filter_item))

    cdx_url = f"https://web.archive.org/cdx/search/cdx?{urlencode(params)}"
    print(f"CDX URL: {cdx_url}")

    print(f"Fetching CDX data for: {url}")
    response = requests.get(cdx_url)
    response.raise_for_status()

    if response.status_code == 200:
        return response.text
    else:
        raise Exception(f"Failed to fetch CDX data for {url}: {response.status_code}")
    


@retry
def download_archived_snapshot(url, timestamp, request_flag="id_", sleep=0.5):
    # Establish a default request frequency of 120 requests/sec
    # This is much lower than the 480/min threshold suggested in the GitHub issue
    # but it does not hurt to be on the safer side here  
    snapshot_url = f"https://web.archive.org/web/{timestamp}{request_flag}/{url}"
    print(f"Fetching archived snapshot for: {snapshot_url}")
    response = requests.get(snapshot_url, allow_redirects=True, stream=True)
    time.sleep(sleep)
    if response.status_code == 404 or response.status_code == 403: 
        return (response.status_code, None, response.headers, response.url)
    response.raise_for_status()
    # detect type of content
    content_type = response.headers.get("Content-Type")
    if (not content_type.startswith("text")):
        return (response.status_code, response.content, response.headers, response.url)
    # if the content is text, set the encoding to apparent encoding to ensure that the text is decoded correctly
    response.encoding = response.apparent_encoding
    return (response.status_code, response.text, response.headers, response.url)


def ensure_absolute_url(src, base_url):
    # Process the source URL to ensure it is absolute.
    # If the src is relative, it will be made absolute using the base URL.
    base_url = f'http://{base_url}' if not base_url.startswith(('http://')) else base_url
    if not (urlparse(src).netloc):
        # If src is relative, make it absolute using the base URL
        src = urljoin(base_url, src)
    elif src.startswith('//'):
        # If src is protocol-relative, add the HTTP scheme
        src = f"http:{src}"
    return src



In [ ]:
### New utils ###
from PIL import Image
from io import BytesIO
import requests
import time


@retry
def download_cdx_closest_data(url, timestamp, sleep=1.5):
    time.sleep(sleep)
    cdx_url = f"https://web.archive.org/cdx/search/cdx?limit=1&url={url}&closest={timestamp}"
    print(f"CDX URL: {cdx_url}")
    response = requests.get(cdx_url)
    response.raise_for_status()

    rows = [row for row in response.text.strip().split("\n") if row]
    if len(rows) == 0:
        return None
    return rows[0]


def check_banner_properties(width: int, height: int) -> dict:
    """Reference to the IAB and JIAA banner ad sizes in banner-ad-dimensions.csv"""
    IAB_SIZES = {
        (88, 31): "Micro Button",
        (120, 240): "Vertical Banner",
        (120, 90): "Button 1",
        (120, 60): "Button 2",
        (125, 125): "Square Button",
        (234, 60): "Half Banner",
        (392, 72): "Full Banner with Vertical Navigation Bar",
        (468, 60): "Full Banner",
    }

    JIAA_SIZES = {
        (120, 90): "Regular Badge",
        (120, 60): "Small Badge",
        (120, 600): "Regular Skyscraper",
        (125, 125): "Large Badge",
        (148, 800): "Large Skyscraper",
        (160, 600): "Wide Skyscraper",
        (200, 200): "Small Rectangle",
        (224, 33): "Small Banner",
        (300, 250): "Regular Rectangle",
        (336, 280): "Large Rectangle",
        (468, 60): "Regular Banner",
        (728, 90): "Large Banner",
    }

    iab_size = IAB_SIZES.get((width, height), None)
    jiaa_size = JIAA_SIZES.get((width, height), None)
    is_banner_ad = iab_size is not None or jiaa_size is not None
    return {
        "iab_size": iab_size,
        "jiaa_size": jiaa_size,
        "is_banner_ad": is_banner_ad,
    }

def get_image_metadata(image_bytes):
    metadata = {
        "width": None,
        "height": None,
        "size": None,
        "animated": None,
        "frame_count": None,
        "animation_duration": None,
        "loop_count": None,
        "iab_size": None,
        "jiaa_size": None,
        "corrupt": True,
    }
    try:
        bytes_io = BytesIO(image_bytes)
        with Image.open(bytes_io) as img:
            metadata = {
                "width": img.width,
                "height": img.height,
                "size": bytes_io.getbuffer().nbytes,
                "animated": False,
                "frame_count": 1,
                "animation_duration": 0,
                "loop_count": 0,
                "iab_size": None,
                "jiaa_size": None,
                "corrupt": False,
            }

            # Check for GIF animation
            if img.format == "GIF" and "duration" in img.info:
                try:
                    metadata["animated"] = True
                    metadata["frame_count"] = img.n_frames
                    metadata["animation_duration"] = (
                        img.info.get("duration", 0) * img.n_frames
                    )
                    metadata["loop_count"] = img.info.get("loop", 0)
                except (AttributeError, KeyError):
                    pass
            # Check if image is a banner ad
            banner_metadata = check_banner_properties(
                metadata["width"], metadata["height"]
            )

            metadata["iab_size"] = banner_metadata["iab_size"]
            metadata["jiaa_size"] = banner_metadata["jiaa_size"]
    except Exception as e:
        print(f"Error getting image metadata: {e}")

    return metadata


In [ ]:
### Create database ###
import sqlite3
DB_NAME = "wm_scraping.db"

conn = sqlite3.connect(DB_NAME)

# source urls table (urls to scrape)
conn.execute(
    "CREATE TABLE IF NOT EXISTS source_urls (url TEXT, has_queried_cdx BOOLEAN DEFAULT FALSE, PRIMARY KEY (url))"
)

# cdx entries for all urls (HTML / media files)
# `is_source_url` is used to indicate if the url is from the source url table
# `source_url` is used to record the url of the source url
# the data structure reflects the CDX API response
conn.execute(
    "CREATE TABLE IF NOT EXISTS cdx_entries (urlkey TEXT, timestamp DATETIME, original TEXT, mimetype TEXT, statuscode INTEGER, digest TEXT, length INTEGER, is_source_url BOOLEAN, source_url TEXT NULL, has_scraped_for_resources BOOLEAN DEFAULT FALSE, PRIMARY KEY (original, timestamp, digest))"
)

# snapshot files (the actual snapshot files)
conn.execute(
    "CREATE TABLE IF NOT EXISTS snapshot_files (digest TEXT, mimetype TEXT, statuscode INTEGER, length INTEGER, file BLOB, md5 TEXT, metadata JSON, PRIMARY KEY (digest))"
)

# snapshot resources table (to record the parent-child relationship between source urls and resources at a given timestamp)
conn.execute(
    "CREATE TABLE IF NOT EXISTS snapshot_resources (parent_url TEXT, parent_timestamp DATETIME, parent_digest TEXT, child_tag TEXT, child_url TEXT, child_tag_attrs JSON, child_digest TEXT NULL,child_timestamp DATETIME NULL, PRIMARY KEY (parent_url, parent_timestamp, parent_digest, child_tag, child_url))"
)
conn.commit()

conn.close()


In [ ]:
### Seed the source urls table ###
import csv, sqlite3

conn = sqlite3.connect(DB_NAME)


with open("nikkeibp-may2000-abridged.csv", "r") as file:
    reader = csv.reader(file)
    next(reader)  # Skip header row
    for row in reader:
        url = row[1]
        conn.execute(
        "INSERT OR IGNORE INTO source_urls (url) VALUES (?)", (url,)
        )

conn.commit()
conn.close()


In [ ]:
### CDX for source urls ###

conn = sqlite3.connect(DB_NAME)

cursor = conn.execute(
    "SELECT url FROM source_urls WHERE has_queried_cdx = FALSE"
)
columns = [desc[0] for desc in cursor.description]
source_urls = [dict(zip(columns, row)) for row in cursor.fetchall()]

cursor.close()

for source_url in source_urls:
    source_url = source_url["url"]
    print(f"Querying CDX entries for {source_url}")

    cdx_entries = download_cdx_data(
        source_url,
        from_timestamp="20000501000000",
        to_timestamp="20000531235959",
        filters=["statuscode:200"],
        collapse="digest",
    )

    cdx_entries = [entry for entry in cdx_entries.strip().split("\n") if entry]

    # Insert all CDX entries for this url
    for cdx_entry in cdx_entries:
        # Parse the CDX entry
        cdx_entry_fields = cdx_entry.split(" ")
        conn.execute(
            "INSERT OR IGNORE INTO cdx_entries (urlkey, timestamp, original, mimetype, statuscode, digest, length, is_source_url, source_url) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)",
            (
                cdx_entry_fields[0],
                cdx_entry_fields[1],
                cdx_entry_fields[2],
                cdx_entry_fields[3],
                cdx_entry_fields[4],
                cdx_entry_fields[5],
                cdx_entry_fields[6],
                True,
                source_url,
            ),
        )

    print(f"Added {len(cdx_entries)} CDX entries for {source_url}")

    # Update the source url status, so that we do not query the same url again
    conn.execute(
        "UPDATE source_urls SET has_queried_cdx = TRUE WHERE url = ?",
        (source_url,),
    )

    # Commit the transaction for this url
    conn.commit()
    print(f"Updated source url {source_url} to has_queried_cdx = TRUE")


conn.close()

In [ ]:
### Download snapshot files for source urls ###
import hashlib

conn = sqlite3.connect(DB_NAME)

# get cdx entries, where it's digest is not in the snapshot_files table
cursor = conn.execute(
    "SELECT * FROM cdx_entries WHERE is_source_url = TRUE AND digest NOT IN (SELECT digest FROM snapshot_files) GROUP BY digest"
)
columns = [desc[0] for desc in cursor.description]
rows = [dict(zip(columns, row)) for row in cursor.fetchall()]
cursor.close()

print(f"Found {len(rows)} source urls to download")

for row in rows:
    original = row['original']
    timestamp = row['timestamp']
    digest = row['digest']
    html_content = download_archived_snapshot(original, timestamp, request_flag="id_", sleep=0.5)
    if html_content == "skip":
        print(f"Skipping download for {original} at {timestamp} due to previous failure.")
        continue
    if html_content[0] != 200: # This is unlikely to happen, but if it does, we will raise an exception
        raise ValueError(f"Failed to download snapshot for {original} at {timestamp}: {html_content[0]}")
    # Save the HTML content to a file named after the timestamp of the snapshot
    html_file_bytes = html_content[1].encode('utf-8')

    md5 = hashlib.md5(html_file_bytes).hexdigest()
   
    conn.execute(
        "INSERT INTO snapshot_files (digest, mimetype, statuscode, length, file, md5) VALUES (?, ?, ?, ?, ?, ?)",
        (digest, html_content[2].get("Content-Type"), html_content[0], len(html_file_bytes), html_file_bytes, md5)
    )
    
    conn.commit()

conn.close()

In [ ]:
### LOOP STEP 1: Extract resources for all downloaded HTML files in cdx_entries ###
from bs4 import BeautifulSoup
def extract_img_tags(html_content, html_base_url):
    soup = BeautifulSoup(html_content, 'html.parser')
    extracted_imgs = []

    img_tags = soup.find_all('img', src=True)
    for img in img_tags:
        src = img.get('src')
        src = ensure_absolute_url(src, html_base_url)
        img['html_base_url'] = html_base_url
        ad_link = img.parent.get('href') if img.parent.name == 'a' else None
        if ad_link:
            ad_link = ensure_absolute_url(ad_link, html_base_url)
        attrs = img.attrs
        attrs["ad_link"] = ad_link
        extracted_imgs.append({
            "tag": "img",
            "url": src,
            "attrs": attrs
        })
     
    return extracted_imgs

def extract_frame_tags(html_content, html_base_url):
    soup = BeautifulSoup(html_content, 'html.parser')
    extracted_frames = []

    frame_tags = soup.find_all('frame', src=True)
    for frame in frame_tags:
        src = frame.get('src')
        src = ensure_absolute_url(src, html_base_url)
        extracted_frames.append({
            "tag": "frame",
            "url": src,
            "attrs": frame.attrs
        })
    
    return extracted_frames

import sqlite3, json
conn = sqlite3.connect(DB_NAME)

cursor = conn.execute(
    'SELECT * FROM cdx_entries JOIN snapshot_files ON cdx_entries.digest = snapshot_files.digest WHERE cdx_entries.mimetype="text/html"'
)

columns_names = [desc[0] for desc in cursor.description]
html_cdx_entries = [dict(zip(columns_names, row)) for row in cursor.fetchall()]
cursor.close()


print(f"Found {len(html_cdx_entries)} source urls to scrape resources for")

for html_cdx_entry in html_cdx_entries:
    original = html_cdx_entry['original']
    timestamp = html_cdx_entry['timestamp'] 
    print(f"Processing {original} at {timestamp}")
    file_blob = html_cdx_entry['file']
    extracted_imgs = extract_img_tags(file_blob, original)
    extracted_frames = extract_frame_tags(file_blob, original)
    try:
      for resource in extracted_imgs + extracted_frames:
          conn.execute(
              "INSERT OR IGNORE INTO snapshot_resources (parent_url, parent_timestamp, parent_digest, child_tag, child_url, child_tag_attrs) VALUES (?, ?, ?, ?, ?, ?)",
              (original, timestamp, html_cdx_entry['digest'], resource['tag'], resource['url'], json.dumps(resource['attrs']))
          )
      conn.execute(
          "UPDATE cdx_entries SET has_scraped_for_resources = TRUE WHERE original = ? AND timestamp = ?",
          (original, timestamp)
      )
    except Exception as e:
        print(f"Error processing {original} at {timestamp}: {e}")
        conn.rollback()
        continue
    finally:
        conn.commit()
conn.close()

In [ ]:
### LOOP STEP 2: CDX for resource urls ###

conn = sqlite3.connect(DB_NAME)
cursor = conn.execute(
    "SELECT * FROM snapshot_resources WHERE child_digest IS NULL ORDER BY child_tag, RANDOM()"
)
columns_names = [desc[0] for desc in cursor.description]
snapshot_resources_without_cdx = [dict(zip(columns_names, row)) for row in cursor.fetchall()]

cursor.close()

for snapshot_resource in snapshot_resources_without_cdx:
    parent_url = snapshot_resource["parent_url"]
    parent_timestamp = snapshot_resource["parent_timestamp"]
    parent_digest = snapshot_resource["parent_digest"]
    child_tag = snapshot_resource["child_tag"]
    child_url = snapshot_resource["child_url"]
    child_tag_attrs = snapshot_resource["child_tag_attrs"]

    cdx_entry = download_cdx_closest_data(child_url, parent_timestamp)
    if cdx_entry:
        try:
            cdx_entry = cdx_entry.split(" ")
            child_digest = cdx_entry[5]
            child_timestamp = cdx_entry[1]
            conn.execute(
                "UPDATE snapshot_resources SET child_digest = ?, child_timestamp = ? WHERE parent_url = ? AND parent_timestamp = ? AND child_tag = ? AND child_url = ?",
                (child_digest, child_timestamp, parent_url, parent_timestamp, child_tag, child_url)
            )

            conn.execute(
                "INSERT OR IGNORE INTO cdx_entries (urlkey, timestamp, original, mimetype, statuscode, digest, length, is_source_url, source_url) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)",
                (cdx_entry[0], cdx_entry[1], cdx_entry[2], cdx_entry[3], cdx_entry[4], cdx_entry[5], cdx_entry[6], False, child_url)
            )
            print(f"Updated child digest for {parent_url} at {parent_timestamp} with {child_digest}")
        except Exception as e:
            print(f"Error updating child digest for {parent_url} at {parent_timestamp}: {e}")
            conn.rollback()
            continue
        finally:
            conn.commit()
    else:
        print(f"No CDX entry found for {child_url} at {parent_timestamp}")

conn.close()

In [ ]:
### LOOP STEP 3: Download resources snapshot files ###
import sqlite3
import hashlib
import json

conn = sqlite3.connect(DB_NAME)

cursor = conn.execute("SELECT * FROM cdx_entries WHERE is_source_url = FALSE AND digest NOT IN (SELECT digest FROM snapshot_files) GROUP BY digest ORDER BY RANDOM()")
column_names = [desc[0] for desc in cursor.description]
undownloaded_cdx_entries = [dict(zip(column_names, row)) for row in cursor.fetchall()]

print(f"Downloading {len(undownloaded_cdx_entries)} snapshot files")
cursor.close()

for cdx_entry in undownloaded_cdx_entries:
    original = cdx_entry['original']
    timestamp = cdx_entry['timestamp']
    digest = cdx_entry['digest']
    request_flag = "id_"

    is_mimetype_image = cdx_entry['mimetype'].startswith('image/') or cdx_entry['mimetype'].startswith('im') 
    is_extension_image = original.endswith('.jpg') or original.endswith('.jpeg') or original.endswith('.png') or original.endswith('.gif') or original.endswith('.webp')
    is_image = is_mimetype_image or is_extension_image
    if is_image:
        request_flag = "im_"
        
    content = download_archived_snapshot(original, timestamp, request_flag=request_flag, sleep=0.5)
    if content == "skip":
        print(f"Skipping download for {original} at {timestamp} due to previous failure.")
        continue
    if content[0] != 200:
        print(f"Failed to download snapshot for {original} at {timestamp}: {content[0]}")
        continue
    
    if isinstance(content[1], str):
        file_bytes = content[1].encode('utf-8')
    else:
        file_bytes = content[1]
        
    md5 = hashlib.md5(file_bytes).hexdigest()
    if is_image:
        image_metadata = get_image_metadata(file_bytes)
    else:
        image_metadata = None
    try:
      conn.execute("INSERT INTO snapshot_files (digest, mimetype, statuscode, length, file, md5, metadata) VALUES (?, ?, ?, ?, ?, ?, ?)", 
      (digest, content[2].get("Content-Type"), content[0], len(file_bytes), file_bytes, md5, json.dumps(image_metadata)))
      conn.execute("UPDATE cdx_entries SET has_scraped_for_resources = TRUE WHERE digest = ?", (digest,))
    except Exception as e:
        print(f"Error inserting snapshot file for {original} at {timestamp}: {e}")
        conn.rollback()
        continue
    finally:  
      conn.commit()
    print(f"Downloaded snapshot for {original} at {timestamp}")

conn.close()